# Module 1 → Tianjin Data Prep

Unzips the Tianjin longitudinal dataset (`usama10/retinal-dr-longitudinal` on Hugging Face),
resolves which eye (OD/OS) each baseline image is (Organized_Data of Patients.xlsx grades DR
per eye, not per patient, and neither corrected_manifest.csv nor the filenames say which eye a
row is), sanity-checks the result against `tianjin_dataset.py`'s loader, and runs the trained
Module 1 classifier + segmentation models over its baseline images to build a third
`module1_cache` (alongside FIRE's and LongDR's from notebook 04), so `train_module2_poc.py`
can condition on real Module 1 masks for Tianjin too -- see
`notebooks/06_module2_poc_with_tianjin_colab.ipynb`.

**Manual prerequisite (one-time):** the Tianjin dataset does not ship as a single official
zip. Download it from
[huggingface.co/datasets/usama10/retinal-dr-longitudinal](https://huggingface.co/datasets/usama10/retinal-dr-longitudinal)
(you may need to accept the license click-through and use a Hugging Face auth token), zip the
extracted contents as `retinal-dr-longitudinal.zip` so that unzipping it produces
`baseline fundus images/`, `2 year follow-up fundus images/`, `Organized_Data of
Patients.xlsx`, and `corrected_manifest.csv` directly, and upload it to
`My Drive/Thesis_Datasets/` alongside the other four archives.

Run this notebook after notebook 04 (it reuses the same Module 1 checkpoints notebook 04
locates on Drive).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
# This repo is PRIVATE -- Colab needs a GitHub Personal Access Token to clone it.
# Create one (once, reusable across all notebooks/sessions) at
# https://github.com/settings/tokens -> "Generate new token (classic)" -> scope: repo.
# Input is hidden; not saved anywhere.
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

# NOTE: this repo now lives at Mieka068/DRProgression (notebooks 01-04 still reference an
# older bearawr/M2-DRProgression location -- fix those the same way if you re-run them).
# Set REPO_BRANCH to whichever branch has the Tianjin loader if it hasn't been merged to
# main yet.
REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'main'  # <-- change if the code you need is on a different branch
# The actual project code lives one level down from the repo root, in this subfolder:
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
TIANJIN_ZIP = os.path.join(DRIVE_DATA_DIR, 'retinal-dr-longitudinal.zip')
assert os.path.isfile(TIANJIN_ZIP), (
    f"Not found: {TIANJIN_ZIP} -- see this notebook's first cell for how to obtain and "
    "upload it, or check the shared Drive folder in case a teammate already staged it."
)

Mounted at /content/drive
GitHub Personal Access Token (repo scope): ··········


In [3]:
# Unzip the Tianjin archive locally on the Colab VM disk (fast local I/O, same convention as
# notebook 01/04 for the other four archives). Robust to whether the zip has a top-level
# wrapper folder or not, by locating corrected_manifest.csv wherever it actually lands and
# normalizing the rest relative to it.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$TIANJIN_ZIP" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, (
    'No corrected_manifest.csv found anywhere under the extracted zip. Run '
    '`!find /content/data/_tianjin_extract_raw -maxdepth 3` to see what actually unzipped, '
    'and check the zip was built correctly (see this notebook\'s first cell).'
)
_tianjin_root = os.path.dirname(_manifest_candidates[0])
print('Found Tianjin dataset root at:', _tianjin_root)

TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR:
    if not os.path.exists(TIANJIN_DIR):
        os.symlink(_tianjin_root, TIANJIN_DIR)
    print(f'Normalized path: {TIANJIN_DIR} -> {_tianjin_root}')

for _expected in ['baseline fundus images', '2 year follow-up fundus images',
                   'Organized_Data of Patients.xlsx', 'corrected_manifest.csv']:
    _p = os.path.join(TIANJIN_DIR, _expected)
    print(('✓ ' if os.path.exists(_p) else '✗ MISSING: ') + _p)

/content/data
Found Tianjin dataset root at: /content/data/_tianjin_extract_raw/retinal-dr-longitudinal
Normalized path: /content/data/retinal-dr-longitudinal -> /content/data/_tianjin_extract_raw/retinal-dr-longitudinal
✓ /content/data/retinal-dr-longitudinal/baseline fundus images
✓ /content/data/retinal-dr-longitudinal/2 year follow-up fundus images
✓ /content/data/retinal-dr-longitudinal/Organized_Data of Patients.xlsx
✓ /content/data/retinal-dr-longitudinal/corrected_manifest.csv


In [4]:
# Clone this repo (private -- uses the token above). Skips cleanly if already cloned in this
# runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch opencv-python-headless

/content
Cloning into 'DRProgression'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 154 (delta 74), reused 80 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.15 MiB | 11.45 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.5 MB/s eta 0:00:00


In [5]:
# Laterality resolution: Organized_Data of Patients.xlsx grades DR per eye (OS/OD), not per
# patient, and corrected_manifest.csv doesn't say which eye a row is -- tianjin_dataset.py /
# module3/dataset.py both require laterality_resolved.csv to exist in TIANJIN_DIR before they
# can run. Reuse a Drive-persisted copy if one already exists (resolving ~1,100 images takes a
# few minutes); otherwise compute it fresh here and persist it to Drive for next time.
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'laterality_resolved.csv')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    import shutil
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive: {LATERALITY_DRIVE_PATH}')
else:
    %cd {REPO_CODE_DIR}
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"
    # Read the printed same-eye disagreement rate above before trusting this for training --
    # see resolve_eye_laterality.py's module docstring (>5% disagreement means the heuristic
    # needs tuning).
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    import shutil
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive: {LATERALITY_DRIVE_PATH}')

/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
Resolved 856/1115 eyes (259 uncertain -- see fallback policy in tianjin_dataset.py)
Wrote /content/data/retinal-dr-longitudinal/laterality_resolved.csv
Patients with 2 baseline images resolving to the SAME eye (should be rare -- flags heuristic problems): 38/543 (7.0%)
⚠ Same-eye disagreement rate is above 5% -- try a larger --margin or smaller --downscale and re-run before using this output for training.
✓ Computed and persisted laterality_resolved.csv to Drive: /content/drive/MyDrive/Thesis_Datasets/module1_cache/laterality_resolved.csv


In [6]:
# Sanity-check corrected_manifest.csv and Organized_Data of Patients.xlsx actually load
# against tianjin_dataset.py's column-name assumptions. This is the first real run against
# the downloaded dataset -- tianjin_dataset.py fails loudly (not silently) if a sheet/column
# name doesn't match, per its own docstring, and prints which columns it matched so you can
# eyeball them.
%cd {REPO_CODE_DIR}
!python tianjin_dataset.py --dataset-dir "{TIANJIN_DIR}"

/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
Testing Tianjin Longitudinal Data Loader
  Reading grades from sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', OD <- 'OD Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', worse-eye <- 'At-risk Eye Grade（Worse eye）'
  Eye-specific grade resolved for 856/1115 rows (259 fell back to the worse-eye summary)
  Excluded grades {6, 7}: 1115 -> 1098 pairs
✓ Found 1098 pairs in Tianjin dataset (1098 with real grade)

Total pairs: 1098
  Baseline shape: torch.Size([3, 128, 128])
  Follow-up shape: torch.Size([3, 

In [7]:
# Locate the same Module 1 checkpoints notebook 04 already found on Drive.
import glob

SEG_DIR   = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation'
CLS_SAVES = '/content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves'

_pref  = ['final_weights.pt', 'best_validation_weights.pt']
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02 first"

EX_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_ex', 'model_2.pth.tar')
MA_CKPT = os.path.join(SEG_DIR, 'models_FGADR_NO_TATL_ma', 'model_2.pth.tar')
for _p in (CLS_CKPT, EX_CKPT, MA_CKPT):
    assert os.path.isfile(_p), f"missing checkpoint: {_p} -- run notebooks 02/03 first"
print('classifier :', CLS_CKPT)
print('EX seg     :', EX_CKPT)
print('MA seg     :', MA_CKPT)

os.makedirs('/content/drive/MyDrive/Thesis_Datasets/module1_cache', exist_ok=True)

classifier : /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves/final_weights.pt
EX seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/models_FGADR_NO_TATL_ex/model_2.pth.tar
MA seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/segmentation/models_FGADR_NO_TATL_ma/model_2.pth.tar


In [8]:
# Run Module 1 over Tianjin's BASELINE images only (not follow-up -- Module 2's conditioning
# input is the baseline image's stage, same convention as FIRE/LongDR). --images-dir is
# pointed at the "baseline fundus images" subfolder specifically, so the cache's image_id
# scheme (relpath from --images-dir, "__"-joined) matches what tianjin_dataset.py's
# _module1_image_id_candidates() looks up -- see that function's docstring if this ever needs
# to change.
%cd {REPO_CODE_DIR}/module1
!python apply_to_progression_data.py \
    --images-dir "{TIANJIN_DIR}/baseline fundus images" \
    --classifier-checkpoint "{CLS_CKPT}" \
    --seg-checkpoint EX="{EX_CKPT}" \
    --seg-checkpoint MA="{MA_CKPT}" \
    --out /content/drive/MyDrive/Thesis_Datasets/module1_cache/module1_outputs_tianjin.pt

/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1
⚠ 00194__._00194-7254: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7254.jpg')
⚠ 00194__._00194-7256: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7256.jpg')
⚠ 00194__._00194-7258: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7258.jpg')
⚠ 00194__._00194-7265: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7265.jpg')
✓ 00194__00194-7256: grade=3 lbs=0.00000 (lesions: ['EX', 'MA'])
✓ 00194__00194-7261: grade=3 lbs=0.00000 (lesions: ['EX', 'MA'])
⚠ 00197__._00197-7299: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00197/._00197-7299.jpg')
⚠ 00197__._00197-7302: failed (cannot identify image file '/content/data/r

## Caveats

- Segmentation-driven conditioning covers EX + MA only, same limitation as FIRE/LongDR in
  notebook 04 -- HE/SE aren't trained yet.
- Tianjin's own clinical grade (from `Organized_Data of Patients.xlsx`) is what conditions
  Module 2 for this source, not the Module 1 classifier's prediction above -- this cache is
  only consulted for the lesion mask channel (`tianjin_dataset.py`'s `module1_cache_path`
  argument, see its docstring).
- The ETDRS(1-5) -> ICDR(0-4) grade mapping is a documented assumption (subtract 1), not a
  verified clinical concordance (see `tianjin_dataset.py`'s module docstring).
- Tianjin is CC BY-NC-4.0 -- non-commercial research use only, no redistribution of the raw
  images outside the dataset's own hosting.
- Next step: `notebooks/06_module2_poc_with_tianjin_colab.ipynb` runs
  `train_module2_poc.py` with all three sources (FIRE + LongDR + Tianjin).